In [12]:
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timedelta
import random
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix,
    precision_score, recall_score
)
from sklearn.ensemble import RandomForestClassifier

### Generating synthetic data

In [13]:
np.random.seed(42)
random.seed(42)

n_rows = 500  
start_time = datetime(2025, 1, 1, 0, 0, 0)
timestamps = [start_time + timedelta(minutes=int(np.random.randint(0, 60*24*30)))
              for _ in range(n_rows)]

users = ["Raghav", "Farhaan", "Naveen", "Navaneeth", "Muhilan", "Mithun", "Betu", "Taru"]
benign_names = [
    "chrome.exe", "firefox.exe", "explorer.exe", "teams.exe", "word.exe",
    "excel.exe", "spotify.exe", "discord.exe", "vscode.exe", "python.exe",
    "outlook.exe", "notepad.exe"
]
suspicious_names = [
    "keylog.exe", "kl_service.exe", "syskey32.exe", "hid_hook.exe",
    "keycap.exe", "keystreamer.exe"
]

process_ids = np.arange(10000, 10000 + n_rows)
chosen_users = np.random.choice(users, size=n_rows, replace=True)

process_name_choices, labels, keystroke_hooks, network_activities, cpu_usages, memory_usages = [], [], [], [], [], []

for _ in range(n_rows):
    if np.random.rand() < 0.20:  # suspicious processes
        pname = np.random.choice(suspicious_names)
        hook = np.random.binomial(1, 0.8)
        net = np.random.binomial(1, 0.75)
        cpu = max(0, np.random.normal(loc=8, scale=5))
        mem = max(10, np.random.normal(loc=120, scale=50))
        base_label = np.random.binomial(1, 0.8)
    else:  # benign processes
        pname = np.random.choice(benign_names)
        hook = np.random.binomial(1, 0.05)
        net = np.random.binomial(1, 0.3)
        cpu = max(0, np.random.normal(loc=22, scale=12))
        mem = max(10, np.random.normal(loc=350, scale=150))
        base_label = np.random.binomial(1, 0.05)

    if np.random.rand() < 0.07:  # label noise
        base_label = 1 - base_label

    process_name_choices.append(pname)
    labels.append(base_label)
    keystroke_hooks.append(hook)
    network_activities.append(net)
    cpu_usages.append(min(cpu, 100))
    memory_usages.append(mem)

df = pd.DataFrame({
    "timestamp": timestamps,
    "process_id": process_ids,
    "user": chosen_users,
    "process_name": process_name_choices,
    "cpu_usage": np.round(cpu_usages, 2),
    "memory_usage": np.round(memory_usages, 1),
    "keystroke_hook": keystroke_hooks,
    "network_activity": network_activities,
    "label": labels
})

# Add missing values
for col in ["cpu_usage", "memory_usage", "user", "process_name"]:
    ix = np.random.choice(df.index, size=int(0.01 * n_rows), replace=False)
    df.loc[ix, col] = np.nan

csv_path = Path("process_logs.csv")
df.to_csv(csv_path, index=False)
print(f"Dataset saved to {csv_path}")

Dataset saved to process_logs.csv


### Load and explore data

In [14]:
df_loaded = pd.read_csv(csv_path)
print("\n--- Data Sample ---")
print(df_loaded.head())

print("\n--- CPU Usage Stats ---")
print(df_loaded["cpu_usage"].describe())

print("\n--- Memory Usage Stats ---")
print(df_loaded["memory_usage"].describe())

hook_count = int(df_loaded["keystroke_hook"].fillna(0).sum())
print(f"\nKeystroke hook enabled count: {hook_count}")


--- Data Sample ---
             timestamp  process_id       user    process_name  cpu_usage  \
0  2025-01-11 23:15:00       10000  Navaneeth  kl_service.exe       9.47   
1  2025-01-01 14:20:00       10001     Raghav       excel.exe      24.62   
2  2025-01-27 11:58:00       10002    Muhilan       teams.exe      16.71   
3  2025-01-08 20:04:00       10003    Muhilan      python.exe       0.00   
4  2025-01-05 08:25:00       10004     Raghav       excel.exe      14.89   

   memory_usage  keystroke_hook  network_activity  label  
0          95.6               1                 1      1  
1         486.6               0                 0      0  
2         587.3               0                 0      0  
3         346.8               0                 0      1  
4         220.4               0                 0      0  

--- CPU Usage Stats ---
count    495.000000
mean      19.537515
std       12.193780
min        0.000000
25%       10.130000
50%       18.210000
75%       27.560000
max

### Preprocess the data

In [15]:
df_prep = df_loaded.copy()
df_prep["timestamp"] = pd.to_datetime(df_prep["timestamp"], errors="coerce")
df_prep["hour"] = df_prep["timestamp"].dt.hour

# Fill missing values
for col in ["cpu_usage", "memory_usage", "hour"]:
    df_prep[col] = df_prep[col].fillna(df_prep[col].median())

for col in ["user", "process_name"]:
    df_prep[col] = df_prep[col].fillna(df_prep[col].mode()[0])

df_prep["keystroke_hook"] = df_prep["keystroke_hook"].fillna(0).astype(int)
df_prep["network_activity"] = df_prep["network_activity"].fillna(0).astype(int)

# Normalize
scaler = StandardScaler()
df_prep[["cpu_usage_norm", "memory_usage_norm"]] = scaler.fit_transform(
    df_prep[["cpu_usage", "memory_usage"]]
)

# One-hot encode
df_model = pd.get_dummies(df_prep, columns=["user", "process_name"], drop_first=True)


### Rule based detection

In [16]:
df_model["rule_flag"] = (
    (df_prep["keystroke_hook"] == 1) &
    (df_prep["network_activity"] == 1) &
    (df_prep["cpu_usage"] < 15.0)
).astype(int)

rule_accuracy = accuracy_score(df_model["label"], df_model["rule_flag"])
rule_precision = precision_score(df_model["label"], df_model["rule_flag"], zero_division=0)
rule_recall = recall_score(df_model["label"], df_model["rule_flag"], zero_division=0)
rule_f1 = f1_score(df_model["label"], df_model["rule_flag"], zero_division=0)
print("\n--- Rule-based Detection Metrics ---")
print(f"Accuracy: {rule_accuracy:.3f}, Precision: {rule_precision:.3f}, Recall: {rule_recall:.3f}, F1: {rule_f1:.3f}")


--- Rule-based Detection Metrics ---
Accuracy: 0.832, Precision: 0.794, Recall: 0.413, F1: 0.543


### Training Random forest classifier

In [17]:
feature_cols = [
    "cpu_usage_norm", "memory_usage_norm", "keystroke_hook", "network_activity", "hour"
] + [c for c in df_model.columns if c.startswith("user_") or c.startswith("process_name_")]

X = df_model[feature_cols]
y = df_model["label"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

rf_accuracy = accuracy_score(y_test, y_pred)
rf_f1 = f1_score(y_test, y_pred)
print("\n--- Random Forest Metrics ---")
print(f"Accuracy: {rf_accuracy:.3f}, F1-score: {rf_f1:.3f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))



--- Random Forest Metrics ---
Accuracy: 0.880, F1-score: 0.727

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.95      0.92        95
           1       0.80      0.67      0.73        30

    accuracy                           0.88       125
   macro avg       0.85      0.81      0.83       125
weighted avg       0.88      0.88      0.88       125



### Saving the alerts

In [18]:
alerts = []

# Rule-based alerts
rule_alerts = df_prep[df_model["rule_flag"] == 1]
for _, row in rule_alerts.iterrows():
    alerts.append({
        "process_id": row["process_id"],
        "process_name": row["process_name"],
        "reason": "Rule-based: keystroke_hook=1, network_activity=1, low CPU"
    })

# Model-based alerts
model_alerts = df_prep.loc[X_test.index][y_pred == 1]
for _, row in model_alerts.iterrows():
    alerts.append({
        "process_id": row["process_id"],
        "process_name": row["process_name"],
        "reason": "Random Forest classified as keylogger"
    })

alerts_df = pd.DataFrame(alerts).drop_duplicates(subset=["process_id", "reason"])
alerts_df.to_csv("alerts.csv", index=False)
print(f"\nAlerts saved to alerts.csv ({len(alerts_df)} alerts)")


Alerts saved to alerts.csv (88 alerts)


### Visualize flagged processes

In [19]:
flagged_users = pd.concat([rule_alerts["user"], model_alerts["user"]], ignore_index=True)
user_counts = flagged_users.value_counts()

plt.figure(figsize=(8,5))
user_counts.plot(kind="bar")
plt.title("Number of Flagged Processes by User")
plt.xlabel("User")
plt.ylabel("Flagged Process Count")
plt.tight_layout()
plt.savefig("flagged_by_user.png")
plt.close()
print("Bar chart saved to flagged_by_user.png")

Bar chart saved to flagged_by_user.png
